In [ ]:
import cv2
import numpy as np
import time

capture_video = cv2.VideoCapture(0)

if not capture_video.isOpened():
    print("Camera not working")
    exit()

time.sleep(2)

background = None

for i in range(60):
    ret, frame = capture_video.read()
    if ret:
        background = frame

if background is None:
    print("Failed to capture background")
    exit()

background = np.flip(background, axis=1)

while True:

    ret, img = capture_video.read()
    if not ret:
        break

    img = np.flip(img, axis=1)

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    lower_red1 = np.array([0,120,70])
    upper_red1 = np.array([10,255,255])
    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)

    lower_red2 = np.array([170,120,70])
    upper_red2 = np.array([180,255,255])
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)

    mask = mask1 + mask2

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3),np.uint8), iterations=2)
    mask = cv2.dilate(mask, np.ones((3,3),np.uint8), iterations=1)

    mask_inv = cv2.bitwise_not(mask)

    res1 = cv2.bitwise_and(background, background, mask=mask)
    res2 = cv2.bitwise_and(img, img, mask=mask_inv)

    final_output = cv2.addWeighted(res1,1,res2,1,0)

    cv2.imshow("Invisible Cloak", final_output)

    if cv2.waitKey(10) == 27:
        break

capture_video.release()
cv2.destroyAllWindows()